In [0]:
%run ./env

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./delta_schema

In [0]:
%run ./load_data

In [0]:
%run ./transform_data

In [0]:
%run ./dim_batch

In [0]:
%run ./dim_parameter

In [0]:
%run ./dim_report_parameter_mapping

In [0]:
%run ./fact_batch_measures

In [0]:
# Controle final : l'equipe energie peut renommer, supprimer ou retirer
# d'energy_metadata un parametre du rapport sans prevenir. Le job a deja tout
# charge a ce stade ; on le fait echouer pour declencher la notification
# d'echec Databricks, sans bloquer le chargement des autres indicateurs.
mapping_cible = spark.table(current_catalog + "." + current_schema + ".dim_report_parameter_mapping")

# Code du mapping different de la source (renomme), parametre supprime ou introuvable
anomalies_codes = (
    mapping_cible.alias("m")
    .join(
        parameters_variables.alias("pv"),
        F.col("m.parameter_id") == F.col("pv.id_parameter_variable"),
        "left"
    )
    .filter(
        F.col("pv.id_parameter_variable").isNull()
        | (F.col("pv.code") != F.col("m.code"))
        | F.coalesce(F.col("pv.deleted"), F.lit(False))
    )
    .select(
        F.col("m.display_order"),
        F.col("m.column_label"),
        F.col("m.code").alias("code_mapping"),
        F.col("pv.code").alias("code_source"),
        F.col("pv.deleted").alias("supprime_source")
    )
)

# Parametre automatique absent d'energy_metadata : ses mesures ne sont plus chargees
anomalies_metadata = (
    mapping_cible
    .filter(~F.col("code").isin(codes_manuels_attendus))
    .join(
        energy_metadata.select(F.col("measure_name").alias("code")).distinct(),
        on="code",
        how="left_anti"
    )
    .select("display_order", "column_label", "code")
)

nb_anomalies_codes = anomalies_codes.count()
nb_anomalies_metadata = anomalies_metadata.count()

if nb_anomalies_codes > 0:
    print(f"ATTENTION : {nb_anomalies_codes} indicateur(s) du rapport dont le code ne correspond plus a la source (renomme ou supprime) :")
    anomalies_codes.orderBy("display_order").show(truncate=False)

if nb_anomalies_metadata > 0:
    print(f"ATTENTION : {nb_anomalies_metadata} indicateur(s) du rapport absents d'energy_metadata, leurs mesures ne sont plus chargees :")
    anomalies_metadata.orderBy("display_order").show(truncate=False)

if nb_anomalies_codes + nb_anomalies_metadata > 0:
    raise Exception(
        "Parametres energie modifies cote source : mettre a jour la liste de "
        "dim_report_parameter_mapping (detail dans la sortie ci-dessus). "
        "Les donnees ont bien ete chargees."
    )